# Paid to Disagree: B8 baseline, synthetic-data anti-sycophancy SFT (Wei et al. 2023)

Fine-tunes each council member (LoRA) on the paper's synthetic claims with a user opinion attached,
teaching it to give the answer it gives without the opinion (`train/b8_sft.py`, `configs/b8.yaml`).
Per member: **filter** (answer the claims without the opinion, keep the ones it gets right) and
**train** (1k LoRA steps, mixed 5:1 with its own answers to our training questions).

**Notebook settings:** Accelerator **GPU T4 x2**, Internet **on**. One session per row:
`MEMBERS = ["qwen", "llama"]` (about 3-4 h), then `MEMBERS = ["gemma"]` (slower, float32).
Output: `b8_runs.zip` with one LoRA per member (`runs/b8_wei_sft/<member>/final`). From the second
session on, attach the previous `b8_runs.zip` as a dataset and set `RESUME_FROM`.

In [ ]:
# ---- settings ----
import time
SESSION_START = time.time()
REPO = "https://github.com/nithish-74/paid-to-disagree.git"
CONFIG = "configs/b8.yaml"
MEMBERS = ["qwen", "llama"]   # second session: ["gemma"]
RESUME_FROM = None            # e.g. "/kaggle/input/b8-runs/runs" from the previous session

## 1. Code and packages

In [ ]:
import os, subprocess, json, pathlib, shutil
os.chdir("/kaggle/working")
if not pathlib.Path("ptd").exists():
    subprocess.run(["git", "clone", "-q", REPO, "ptd"], check=True)
os.chdir("/kaggle/working/ptd")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=True)
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q unsloth
!pip install -q vllm trl
!pip install -q -e .
!python -c "import unsloth, trl, vllm, torch; print('unsloth', unsloth.__version__, '| trl', trl.__version__, '| vllm', vllm.__version__, '| GPUs', torch.cuda.device_count())"

## 2. Model access, data, synthetic prompts

In [ ]:
import yaml
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
cfg = yaml.safe_load(open(CONFIG))
council = yaml.safe_load(open(cfg["council_config"]))
if token:
    os.environ["HF_TOKEN"] = token
else:
    mirrors = {"meta-llama/Llama-3.2-1B-Instruct": "unsloth/Llama-3.2-1B-Instruct",
               "google/gemma-3-1b-it": "unsloth/gemma-3-1b-it"}
    for m in council["members"]:
        m["backend"]["model"] = mirrors.get(m["backend"]["model"], m["backend"]["model"])
    print("No HF_TOKEN secret: using Unsloth's ungated copies of the same weights.")
yaml.safe_dump(council, open("configs/council_kaggle.yaml", "w"), sort_keys=False)
cfg["council_config"] = "configs/council_kaggle.yaml"
yaml.safe_dump(cfg, open("configs/b8_kaggle.yaml", "w"), sort_keys=False)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, "runs", dirs_exist_ok=True)
    print("restored runs/ from", RESUME_FROM)

In [ ]:
!python -m data.build_questions --config configs/data.yaml --sources truthfulqa_mc1 arc_challenge mmlu_pro
if not pathlib.Path(f"runs/{cfg['experiment']}/synthetic.jsonl").exists():
    subprocess.run(["python", "-m", "train.b8_sft", "generate", "--config", "configs/b8_kaggle.yaml"], check=True)
print(sum(1 for _ in open(f"runs/{cfg['experiment']}/synthetic.jsonl")), "synthetic prompts")

## 3. Filter and train each member

In [ ]:
def step(command, member, every=300):
    log = pathlib.Path(f"runs/b8.{member}.{command}.log")
    start = time.time()
    with open(log, "a") as f:
        p = subprocess.Popen(["python", "-m", "train.b8_sft", command, "--config", "configs/b8_kaggle.yaml",
                              "--member", member], stdout=f, stderr=subprocess.STDOUT,
                             env={**os.environ, "PYTHONUNBUFFERED": "1"})
        while p.poll() is None:
            time.sleep(every)
            lines = [l.rstrip() for l in open(log, errors="replace") if l.strip()]
            print(f"[{(time.time() - start) / 60:5.1f} min] {member} {command}: {lines[-1][:200] if lines else '...'}", flush=True)
    lines = [l.rstrip() for l in open(log, errors="replace") if l.strip()]
    print("\n".join(lines[-8:]))
    assert p.returncode == 0, f"{member} {command} failed: see {log}"


exp = pathlib.Path("runs") / cfg["experiment"]
for m in MEMBERS:
    if not (exp / f"{m}.synthetic_sft.jsonl").exists():
        step("filter", m)
    print(json.load(open(exp / f"{m}.filter_stats.json")))
    if not (exp / m / "final").exists():
        step("train", m, every=600)
    print(f"{m} done; session time {(time.time() - SESSION_START) / 3600:.1f} h")

## 4. Package

In [ ]:
out = shutil.make_archive("/kaggle/working/b8_runs", "zip", ".", "runs")
print(out, f"{os.path.getsize(out) / 1e6:.1f} MB")